In [0]:
#bootstrap_servers= 'pkc-9q8rv.ap-south-2.aws.confluent.cloud:9092'
#api_key = 'PXOG3YYPFIXUVR3J'
#api_secret = 'cfltI8fVS3LG5ckd4O97G8CAKd+1EUO8uP9Z98ABZzpK3CVMSvBVNOahbZLx4Y+w'
#topic = 'finguard'

In [0]:
bootstrap_servers = dbutils.secrets.get(scope="kafka_scope",key="bootstrap_servers")
api_key = dbutils.secrets.get(scope="kafka_scope",key="api_key")
api_secret = dbutils.secrets.get(scope="kafka_scope",key="api_secret")
topic = dbutils.secrets.get(scope="kafka_scope",key="topic")

In [0]:
jaas_config=f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{api_key}" password="{api_secret}";'


## Batch Data Processing

In [0]:
sample_batch=(spark.read.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_servers)
              .option("subscribe",topic)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest")
        .load())


In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

In [0]:
from pyspark.sql.functions import col

parshed_batch = sample_batch.select( 
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

In [0]:
display(parshed_batch)

In [0]:
parshed_batch.write.saveAsTable("finguard.bronze.transaction_batch_test")

## Streaming Data Processing

In [0]:
streaming_data=(spark.readStream.format("kafka")
              .option("kafka.bootstrap.servers",bootstrap_servers)
              .option("subscribe",topic)
            .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", jaas_config)
        .option("startingOffsets","earliest")
        .load())

In [0]:
from pyspark.sql.functions import col

parshed_streaming_data = streaming_data.select( 
    col("key").cast("string"),
    col("value").cast("string"),
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp")
)

In [0]:
parshed_streaming_data.writeStream.format("delta")\
.outputMode("append")\
    .option("checkpointLocation","/Volumes/finguard/source/transaction/checkpoint/")\
        .trigger(availableNow=True)\
            .toTable("finguard.bronze.transaction_streaming_test")


In [0]:
%sql
select * from finguard.bronze.transaction_streaming_test